# Smart MCQ Solver — Ensemble (TF-IDF Pairwise Ranker + BiLSTM Pairwise Scorer)

**Goal:** combine the two trained models into one stronger predictor.

| Model | Formulation | Val MAP@3 |
|---|---|---|
| TF-IDF dual-branch pairwise ranker | word+char TF-IDF → MLP → 1 correctness score per option | 0.75311 |
| BiLSTM + attention pairwise scorer | GloVe embeddings → BiLSTM → attention → 1 score per option | 0.74029 |

**Why ensembling should help:** both models are trained on the *same* 5-way ranking problem but
look at the text completely differently — one sees sparse word/char n-gram counts, the other sees
a learned sequential representation. When two models make **different mistakes**, averaging their
opinions cancels out some of the noise and keeps the signal both agree on. This is Ensemble
Strategy 1 ("average probabilities") — simple, and very easy to explain in a viva.

**What this notebook does, in order:**
1. Load data once, split into train/val **by question id**, and reuse the *same* split for both
   models so their validation scores are directly comparable.
2. Train Model A (TF-IDF ranker) — condensed version of the standalone notebook.
3. Train Model B (BiLSTM scorer) — condensed version of the standalone notebook.
4. Turn both models' raw scores into **per-question probability distributions** (softmax over the
   5 options) so they're on the same scale before combining.
5. Grid-search one number — the ensemble weight `alpha` — on the validation set, log everything to
   the same W&B run, and write the final `submission_ensemble.csv`.


### Fix: `AcceleratorError: no kernel image is available for execution on the device`
Same fix as the standalone notebooks: install extra packages with `--no-deps`, and verify the GPU
with a real tensor op instead of trusting `torch.cuda.is_available()` alone.


In [1]:
# Cell 1 — Imports, seeding, device check
!pip install -q --no-deps wandb

import re, glob, random
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
import wandb

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

def get_device():
    if not torch.cuda.is_available():
        return torch.device("cpu")
    try:
        (torch.zeros(1, device="cuda") + 1).cpu()   # real op, not just is_available()
        return torch.device("cuda")
    except Exception as e:
        print("CUDA reports available but a test op failed, falling back to CPU:", e)
        return torch.device("cpu")

device = get_device()
print("Using device:", device)


/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Found GPU0 Tesla P100-PCIE-16GB which is of cuda capability 6.0.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (7.0) - (12.0)
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
    Please install PyTorch with a following CUDA
    configurations:  12.6 following instructions at
    https://pytorch.org/get-started/locally/
    
  queued_call()
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:435: UserWarning: 
Tesla P100-PCIE-16GB with CUDA capability sm_60 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_70 sm_75 sm_80 sm_86 sm_90 sm_100 sm_120.
If you want to use the Tesla P100-PCIE-16GB GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  queued_call()


CUDA reports available but a test op failed, falling back to CPU: CUDA error: no kernel image is available for execution on the device
Search for `cudaErrorNoKernelImageForDevice' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.

Using device: cpu


### Config
One config dict, split into three clearly-labelled groups: `tfidf_*` keys (Model A), `bilstm_*`
keys (Model B), and `ensemble_*` keys (the new part). Nothing here is exotic — every number is
copied from whichever standalone notebook already scored well with it.


In [2]:
# Cell 2 — Config + single shared W&B run for the whole ensemble notebook
CONFIG = {
    # ---- Model A: TF-IDF pairwise ranker ----
    "tfidf_word_max_features": 40000, "tfidf_word_ngram_range": (1, 3),
    "tfidf_word_min_df": 2, "tfidf_word_max_df": 0.9,
    "tfidf_char_max_features": 20000, "tfidf_char_ngram_range": (3, 5),
    "tfidf_char_min_df": 2, "tfidf_char_max_df": 0.95,
    "tfidf_word_hidden": 512, "tfidf_char_hidden": 256, "tfidf_merged_hidden": 256,
    "tfidf_dropout": 0.3, "tfidf_label_smoothing": 0.05,
    "tfidf_lr": 1e-3, "tfidf_weight_decay": 1e-4, "tfidf_batch_size": 256,
    "tfidf_epochs": 40, "tfidf_early_stop_min_epochs": 12,
    "tfidf_early_stop_patience": 8, "tfidf_early_stop_min_delta": 0.001,

    # ---- Model B: BiLSTM + attention pairwise scorer ----
    "bilstm_max_vocab_size": 30000, "bilstm_min_freq": 2,
    "bilstm_embed_dim": 100, "bilstm_hidden_dim": 128, "bilstm_num_layers": 2,
    "bilstm_dropout": 0.35, "bilstm_label_smoothing": 0.05,
    "bilstm_lr": 1e-3, "bilstm_weight_decay": 1e-4, "bilstm_batch_size": 32,
    "bilstm_epochs": 40, "bilstm_grad_clip": 1.0,
    "bilstm_early_stop_min_epochs": 12, "bilstm_early_stop_patience": 10,
    "bilstm_early_stop_min_delta": 0.0005,
    "bilstm_use_pretrained_embeddings": True, "bilstm_freeze_embeddings": False,
    "bilstm_use_synonym_aug": True, "bilstm_aug_fraction": 0.3,

    # ---- Ensemble ----
    "val_split": 0.15,                 # shared train/val split, by question id
    "alpha_grid_step": 0.05,           # ensemble_score = alpha*tfidf + (1-alpha)*bilstm
}

wandb.login()
wandb.init(project="smart-mcq-solver", name="tfidf-bilstm-ensemble", config=CONFIG)


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

  ········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


## 1. Shared data + shared split
**The one rule that makes this ensemble valid:** both models must be validated on the *exact same*
set of question ids. If Model A validates on questions {1,2,3} and Model B validates on
{4,5,6}, their val MAP@3 scores aren't comparable and combining their scores at inference is
meaningless. So we split **once**, on `id`, before either model touches the data.


In [3]:
# Cell 3 — Load once, split once (by question id), shared by both models
train_df = pd.read_csv("/kaggle/input/datasets/kirandwippaul/mcqdata/train.csv")
test_df  = pd.read_csv("/kaggle/input/datasets/kirandwippaul/mcqdata/test.csv")

LABELS = ["A", "B", "C", "D", "E"]
label2idx = {l: i for i, l in enumerate(LABELS)}

for df in (train_df, test_df):
    for col in ["prompt"] + LABELS:
        df[col] = df[col].fillna("").astype(str)

train_df["label"] = train_df["answer"].map(label2idx)

train_qids, val_qids = train_test_split(
    train_df["id"], test_size=CONFIG["val_split"], random_state=SEED,
    stratify=train_df["label"],
)
train_qids, val_qids = set(train_qids), set(val_qids)

assert train_qids.isdisjoint(val_qids), "leakage: a qid is on both sides of the split"
print(f"Train questions: {len(train_qids)} | Val questions: {len(val_qids)}")


Train questions: 1700 | Val questions: 300


## 2. Model A — TF-IDF dual-branch pairwise ranker
Every `(question, option)` pair becomes its own row with a binary label ("is this option
correct?"). Word TF-IDF + char TF-IDF each go through their own small MLP branch, the branches
are concatenated, and one linear layer outputs a single correctness logit. Identical to the
standalone notebook, just filtered onto the shared split.


In [4]:
# Cell 4 — Melt into (question, option) rows + fit TF-IDF vectorizers
def melt_options(df, has_labels=True):
    frames = []
    for letter in LABELS:
        sub = pd.DataFrame({
            "qid": df["id"],
            "option": letter,
            "text": "Question: " + df["prompt"] + " Answer: " + df[letter],
        })
        if has_labels:
            sub["label"] = (df["answer"] == letter).astype(int)
        frames.append(sub)
    return pd.concat(frames, ignore_index=True)

train_long_all = melt_options(train_df)          # every question, 5 rows each
test_long      = melt_options(test_df, has_labels=False)

tfidf_train_long = train_long_all[train_long_all["qid"].isin(train_qids)].reset_index(drop=True)
tfidf_val_long   = train_long_all[train_long_all["qid"].isin(val_qids)].reset_index(drop=True)

word_tfidf = TfidfVectorizer(
    analyzer="word", max_features=CONFIG["tfidf_word_max_features"],
    ngram_range=CONFIG["tfidf_word_ngram_range"], min_df=CONFIG["tfidf_word_min_df"],
    max_df=CONFIG["tfidf_word_max_df"], sublinear_tf=True, stop_words="english",
)
char_tfidf = TfidfVectorizer(
    analyzer="char_wb", max_features=CONFIG["tfidf_char_max_features"],
    ngram_range=CONFIG["tfidf_char_ngram_range"], min_df=CONFIG["tfidf_char_min_df"],
    max_df=CONFIG["tfidf_char_max_df"], sublinear_tf=True,
)

all_text = pd.concat([train_long_all["text"], test_long["text"]])
word_tfidf.fit(all_text)
char_tfidf.fit(all_text)

Xw_train, Xc_train = word_tfidf.transform(tfidf_train_long["text"]), char_tfidf.transform(tfidf_train_long["text"])
Xw_val,   Xc_val   = word_tfidf.transform(tfidf_val_long["text"]),   char_tfidf.transform(tfidf_val_long["text"])
Xw_test,  Xc_test  = word_tfidf.transform(test_long["text"]),        char_tfidf.transform(test_long["text"])
y_train, y_val = tfidf_train_long["label"].values, tfidf_val_long["label"].values

print("Word vocab:", len(word_tfidf.vocabulary_), "| Char vocab:", len(char_tfidf.vocabulary_))
print("TF-IDF train rows:", Xw_train.shape[0], "| val rows:", Xw_val.shape[0])


Word vocab: 24198 | Char vocab: 20000
TF-IDF train rows: 8500 | val rows: 1500


In [5]:
# Cell 5 — Dataset / DataLoader for the TF-IDF model
class DualTfidfDataset(Dataset):
    def __init__(self, X_word, X_char, y=None):
        self.X_word, self.X_char, self.y = X_word, X_char, y

    def __len__(self):
        return self.X_word.shape[0]

    def __getitem__(self, idx):
        w = torch.from_numpy(self.X_word[idx].toarray().squeeze(0)).float()
        c = torch.from_numpy(self.X_char[idx].toarray().squeeze(0)).float()
        if self.y is not None:
            return w, c, torch.tensor(self.y[idx], dtype=torch.float32)
        return w, c

tfidf_train_loader = DataLoader(DualTfidfDataset(Xw_train, Xc_train, y_train), batch_size=CONFIG["tfidf_batch_size"], shuffle=True)
tfidf_val_loader   = DataLoader(DualTfidfDataset(Xw_val, Xc_val, y_val),       batch_size=CONFIG["tfidf_batch_size"], shuffle=False)
tfidf_test_loader  = DataLoader(DualTfidfDataset(Xw_test, Xc_test),           batch_size=CONFIG["tfidf_batch_size"], shuffle=False)


In [6]:
# Cell 6 — TF-IDF model definition
class DualBranchScorer(nn.Module):
    """Dual-branch (word + char) network ending in ONE logit: how correct does this
    (question, option) pair look?"""
    def __init__(self, word_vocab, char_vocab, word_hidden, char_hidden, merged_hidden, dropout=0.3):
        super().__init__()
        self.word_branch = nn.Sequential(
            nn.Linear(word_vocab, word_hidden), nn.BatchNorm1d(word_hidden), nn.GELU(), nn.Dropout(dropout),
        )
        self.char_branch = nn.Sequential(
            nn.Linear(char_vocab, char_hidden), nn.BatchNorm1d(char_hidden), nn.GELU(), nn.Dropout(dropout),
        )
        self.head = nn.Sequential(
            nn.Linear(word_hidden + char_hidden, merged_hidden),
            nn.BatchNorm1d(merged_hidden), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(merged_hidden, 1),
        )

    def forward(self, x_word, x_char):
        w = self.word_branch(x_word)
        c = self.char_branch(x_char)
        return self.head(torch.cat([w, c], dim=1)).squeeze(1)   # (batch,)

tfidf_model = DualBranchScorer(
    word_vocab=Xw_train.shape[1], char_vocab=Xc_train.shape[1],
    word_hidden=CONFIG["tfidf_word_hidden"], char_hidden=CONFIG["tfidf_char_hidden"],
    merged_hidden=CONFIG["tfidf_merged_hidden"], dropout=CONFIG["tfidf_dropout"],
).to(device)

wandb.watch(tfidf_model, log="all", log_freq=50, idx=0)
print(tfidf_model)


DualBranchScorer(
  (word_branch): Sequential(
    (0): Linear(in_features=24198, out_features=512, bias=True)
    (1): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): GELU(approximate='none')
    (3): Dropout(p=0.3, inplace=False)
  )
  (char_branch): Sequential(
    (0): Linear(in_features=20000, out_features=256, bias=True)
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): GELU(approximate='none')
    (3): Dropout(p=0.3, inplace=False)
  )
  (head): Sequential(
    (0): Linear(in_features=768, out_features=256, bias=True)
    (1): BatchNorm1d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): GELU(approximate='none')
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=256, out_features=1, bias=True)
  )
)


### Shared MAP@3 helper
Both models eventually produce one score per `(qid, option)` row. This single function computes
MAP@3 from that shape, so it's reused for Model A validation, Model B validation, *and* the final
ensemble — one metric definition, no room for the two models to be scored inconsistently.


In [7]:
# Cell 7 — Shared MAP@3 metric (works for any model, as long as scores are in this shape)
def compute_map3(df, score_col="score"):
    """df needs columns: qid, option, <score_col>, label (5 rows per qid, one label==1)."""
    scores = []
    for _, g in df.groupby("qid"):
        ranked = g.sort_values(score_col, ascending=False)["option"].tolist()
        true_option = g.loc[g["label"] == 1, "option"].iloc[0]
        top3 = ranked[:3]
        scores.append(1.0 / (top3.index(true_option) + 1) if true_option in top3 else 0.0)
    return sum(scores) / len(scores)


In [8]:
# Cell 8 — Train Model A (TF-IDF), logging to W&B with a "tfidf/" prefix
tfidf_criterion = nn.BCEWithLogitsLoss()
tfidf_optimizer = torch.optim.AdamW(tfidf_model.parameters(), lr=CONFIG["tfidf_lr"], weight_decay=CONFIG["tfidf_weight_decay"])
tfidf_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(tfidf_optimizer, mode="max", factor=0.5, patience=3)
tfidf_scaler = torch.amp.GradScaler(device.type, enabled=(device.type == "cuda"))
eps = CONFIG["tfidf_label_smoothing"]

def tfidf_train_one_epoch():
    tfidf_model.train()
    total_loss, n_batches = 0.0, 0
    for Xw, Xc, y in tfidf_train_loader:
        Xw, Xc, y = Xw.to(device), Xc.to(device), y.to(device)
        y_smooth = y * (1 - eps) + 0.5 * eps
        with torch.amp.autocast(device.type, enabled=(device.type == "cuda")):
            logits = tfidf_model(Xw, Xc)
            loss = tfidf_criterion(logits, y_smooth)
        tfidf_optimizer.zero_grad()
        tfidf_scaler.scale(loss).backward()
        tfidf_scaler.step(tfidf_optimizer)
        tfidf_scaler.update()
        total_loss += loss.item(); n_batches += 1
    return total_loss / n_batches

@torch.no_grad()
def tfidf_evaluate(loader, meta_df):
    tfidf_model.eval()
    total_loss, n_batches, all_scores = 0.0, 0, []
    for Xw, Xc, y in loader:
        Xw, Xc, y = Xw.to(device), Xc.to(device), y.to(device)
        with torch.amp.autocast(device.type, enabled=(device.type == "cuda")):
            logits = tfidf_model(Xw, Xc)
            loss = tfidf_criterion(logits, y)
        total_loss += loss.item(); n_batches += 1
        all_scores.extend(torch.sigmoid(logits).float().cpu().tolist())
    scored = meta_df.copy()
    scored["score"] = all_scores
    return total_loss / n_batches, compute_map3(scored)

best_tfidf_map3, patience_left = 0.0, CONFIG["tfidf_early_stop_patience"]
meta_val_tfidf = tfidf_val_long[["qid", "option", "label"]].reset_index(drop=True)

for epoch in range(1, CONFIG["tfidf_epochs"] + 1):
    train_loss = tfidf_train_one_epoch()
    val_loss, val_map3 = tfidf_evaluate(tfidf_val_loader, meta_val_tfidf)
    tfidf_scheduler.step(val_map3)

    wandb.log({
        "tfidf/epoch": epoch, "tfidf/train_loss": train_loss,
        "tfidf/val_loss": val_loss, "tfidf/val_map3": val_map3,
        "tfidf/lr": tfidf_optimizer.param_groups[0]["lr"],
    })

    if val_map3 > best_tfidf_map3 + CONFIG["tfidf_early_stop_min_delta"]:
        best_tfidf_map3 = val_map3
        patience_left = CONFIG["tfidf_early_stop_patience"]
        torch.save(tfidf_model.state_dict(), "best_tfidf_model.pt")
    elif epoch >= CONFIG["tfidf_early_stop_min_epochs"]:
        patience_left -= 1

    print(f"[TF-IDF] Epoch {epoch:02d} | train_loss {train_loss:.4f} val_loss {val_loss:.4f} | val_map3 {val_map3:.4f} | patience {patience_left}")
    if epoch >= CONFIG["tfidf_early_stop_min_epochs"] and patience_left <= 0:
        print("Early stopping (TF-IDF).")
        break

tfidf_model.load_state_dict(torch.load("best_tfidf_model.pt"))
print("Best TF-IDF val MAP@3:", best_tfidf_map3)


[TF-IDF] Epoch 01 | train_loss 0.4538 val_loss 0.4124 | val_map3 0.9794 | patience 8
[TF-IDF] Epoch 02 | train_loss 0.2182 val_loss 0.1094 | val_map3 0.9911 | patience 8
[TF-IDF] Epoch 03 | train_loss 0.1749 val_loss 0.0759 | val_map3 0.9894 | patience 8
[TF-IDF] Epoch 04 | train_loss 0.1583 val_loss 0.0638 | val_map3 0.9911 | patience 8
[TF-IDF] Epoch 05 | train_loss 0.1495 val_loss 0.0583 | val_map3 0.9911 | patience 8
[TF-IDF] Epoch 06 | train_loss 0.1421 val_loss 0.0540 | val_map3 0.9911 | patience 8
[TF-IDF] Epoch 07 | train_loss 0.1384 val_loss 0.0504 | val_map3 0.9911 | patience 8
[TF-IDF] Epoch 08 | train_loss 0.1360 val_loss 0.0494 | val_map3 0.9911 | patience 8
[TF-IDF] Epoch 09 | train_loss 0.1344 val_loss 0.0470 | val_map3 0.9911 | patience 8
[TF-IDF] Epoch 10 | train_loss 0.1332 val_loss 0.0469 | val_map3 0.9911 | patience 8
[TF-IDF] Epoch 11 | train_loss 0.1314 val_loss 0.0461 | val_map3 0.9911 | patience 8
[TF-IDF] Epoch 12 | train_loss 0.1316 val_loss 0.0457 | val_map3 

## 3. Model B — BiLSTM + attention pairwise scorer
Same pairwise idea, different encoder: each `(question, option)` pair is tokenized, embedded
(starting from GloVe if a GloVe dataset is attached), passed through a BiLSTM with additive
attention, and pooled to one relevance score. Trained on the same shared question split.


In [9]:
# Cell 9 — Tokenizer, vocab, safe synonym augmentation (train side only)
def build_pair_text(question, option):
    return f"question: {question} answer: {option}".lower()

def tokenize(text):
    return re.findall(r"[a-z0-9]+", text)

bilstm_train_part = train_df[train_df["id"].isin(train_qids)].reset_index(drop=True)
bilstm_val_part   = train_df[train_df["id"].isin(val_qids)].reset_index(drop=True)

# max_len chosen empirically (covers ~98% of pair-texts), same approach as the standalone notebook
sample_lengths = [
    len(tokenize(build_pair_text(row.prompt, getattr(row, letter))))
    for row in bilstm_train_part.itertuples() for letter in LABELS
]
CONFIG["bilstm_max_len"] = int(min(max(np.percentile(sample_lengths, 98), 16), 64))
print("max_len set to:", CONFIG["bilstm_max_len"])

SAFE_SYNONYMS = {
    "large": "big", "small": "tiny", "fast": "quick", "begin": "start", "end": "finish",
    "happy": "glad", "sad": "unhappy", "hard": "difficult", "easy": "simple", "big": "large",
    "old": "ancient", "new": "modern", "buy": "purchase", "help": "assist", "show": "display",
}

def synonym_swap(text, rate=0.3):
    words = text.split()
    for i, w in enumerate(words):
        if w in SAFE_SYNONYMS and random.random() < rate:
            words[i] = SAFE_SYNONYMS[w]
    return " ".join(words)

def augment(df, fraction):
    n = int(len(df) * fraction)
    rows = df.sample(n=n, random_state=SEED).copy()
    rows["prompt"] = rows["prompt"].apply(lambda t: synonym_swap(t.lower()))
    return rows

if CONFIG["bilstm_use_synonym_aug"]:
    bilstm_train_part = pd.concat(
        [bilstm_train_part, augment(bilstm_train_part, CONFIG["bilstm_aug_fraction"])], ignore_index=True
    )
print("BiLSTM train size (post-augmentation):", len(bilstm_train_part), "| val size:", len(bilstm_val_part))

def build_vocab(texts, max_vocab_size, min_freq):
    counter = Counter()
    for t in texts:
        counter.update(tokenize(t))
    vocab = {"<PAD>": 0, "<UNK>": 1}
    for token, freq in counter.most_common():
        if freq < min_freq or len(vocab) >= max_vocab_size:
            continue
        vocab[token] = len(vocab)
    return vocab

vocab = build_vocab(
    [build_pair_text(r.prompt, getattr(r, l)) for r in bilstm_train_part.itertuples() for l in LABELS],
    CONFIG["bilstm_max_vocab_size"], CONFIG["bilstm_min_freq"],
)
PAD_IDX, UNK_IDX = vocab["<PAD>"], vocab["<UNK>"]
print("Vocab size:", len(vocab))


max_len set to: 64
BiLSTM train size (post-augmentation): 2210 | val size: 300
Vocab size: 2973


In [10]:
# Cell 10 — GloVe embedding lookup (falls back to random init if no GloVe dataset attached)
def find_glove_file(dim):
    import os
    candidates = [
        f"/kaggle/input/glove6b{dim}dtxt/glove.6B.{dim}d.txt",
        f"/kaggle/input/glove-global-vectors-for-word-representation/glove.6B.{dim}d.txt",
    ]
    candidates += glob.glob(f"/kaggle/input/**/glove.6B.{dim}d.txt", recursive=True)
    for path in candidates:
        if path and os.path.exists(path):
            return path
    return None

def load_glove_matrix(vocab, embed_dim):
    matrix = np.random.normal(scale=0.1, size=(len(vocab), embed_dim)).astype(np.float32)
    matrix[PAD_IDX] = 0.0
    path = find_glove_file(embed_dim)
    if path is None:
        print("No GloVe file found under /kaggle/input -- falling back to random init.")
        return matrix, 0
    hits = 0
    with open(path, encoding="utf-8") as f:
        for line in f:
            parts = line.rstrip().split(" ")
            if parts[0] in vocab:
                matrix[vocab[parts[0]]] = np.asarray(parts[1:], dtype=np.float32)
                hits += 1
    return matrix, hits

if CONFIG["bilstm_use_pretrained_embeddings"]:
    embed_matrix, n_hits = load_glove_matrix(vocab, CONFIG["bilstm_embed_dim"])
    print(f"GloVe coverage: {n_hits}/{len(vocab)} vocab words ({n_hits / len(vocab):.1%})")
else:
    embed_matrix = None


No GloVe file found under /kaggle/input -- falling back to random init.
GloVe coverage: 0/2973 vocab words (0.0%)


In [11]:
# Cell 11 — Encoding, Dataset/DataLoader (one question, 5 options per item)
def encode(text, vocab, max_len):
    tokens = tokenize(text)[:max_len]
    length = len(tokens)
    ids = [vocab.get(tok, UNK_IDX) for tok in tokens]
    ids += [PAD_IDX] * (max_len - length)
    return ids, max(length, 1)

class MCQPairDataset(Dataset):
    def __init__(self, df, vocab, max_len, has_labels=True):
        self.df, self.vocab, self.max_len, self.has_labels = df, vocab, max_len, has_labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        ids_per_option, len_per_option = [], []
        for letter in LABELS:
            ids, length = encode(build_pair_text(row["prompt"], row[letter]), self.vocab, self.max_len)
            ids_per_option.append(ids)
            len_per_option.append(length)
        ids = torch.tensor(ids_per_option, dtype=torch.long)
        lengths = torch.tensor(len_per_option, dtype=torch.long)
        if self.has_labels:
            return ids, lengths, torch.tensor(row["label"], dtype=torch.long)
        return ids, lengths

bilstm_train_loader = DataLoader(MCQPairDataset(bilstm_train_part, vocab, CONFIG["bilstm_max_len"]), batch_size=CONFIG["bilstm_batch_size"], shuffle=True)
bilstm_val_loader   = DataLoader(MCQPairDataset(bilstm_val_part, vocab, CONFIG["bilstm_max_len"]),   batch_size=CONFIG["bilstm_batch_size"], shuffle=False)
bilstm_test_loader  = DataLoader(MCQPairDataset(test_df, vocab, CONFIG["bilstm_max_len"], has_labels=False), batch_size=CONFIG["bilstm_batch_size"], shuffle=False)


In [12]:
# Cell 12 — BiLSTM + attention model definition
class WordEmbedding(nn.Module):
    def __init__(self, vocab_size, embed_dim, pad_idx, pretrained=None, freeze=False):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        if pretrained is not None:
            self.embedding.weight.data.copy_(torch.from_numpy(pretrained))
            self.embedding.weight.requires_grad = not freeze

    def forward(self, x):
        return self.embedding(x)

class BiLSTMEncoder(nn.Module):
    def __init__(self, embed_dim, hidden_dim, num_layers, dropout):
        super().__init__()
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim, num_layers=num_layers, batch_first=True, bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )

    def forward(self, x, lengths):
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_out, _ = self.lstm(packed)
        out, _ = pad_packed_sequence(packed_out, batch_first=True, total_length=x.size(1))
        return out

class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, lstm_out, lengths):
        scores = self.v(torch.tanh(self.attn(lstm_out))).squeeze(-1)
        seq_len = lstm_out.size(1)
        mask = torch.arange(seq_len, device=lengths.device)[None, :] < lengths[:, None]
        scores = scores.masked_fill(~mask, -1e9)
        weights = torch.softmax(scores, dim=1)
        context = torch.bmm(weights.unsqueeze(1), lstm_out).squeeze(1)
        return context, weights

class ScoreHead(nn.Module):
    def __init__(self, hidden_dim, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )

    def forward(self, context):
        return self.net(context).squeeze(-1)

class PairScorer(nn.Module):
    """Scores one (question, option) pair at a time -- called once per option, 5x per question."""
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers, dropout, pad_idx, pretrained=None, freeze=False):
        super().__init__()
        self.embedding = WordEmbedding(vocab_size, embed_dim, pad_idx, pretrained, freeze)
        self.encoder = BiLSTMEncoder(embed_dim, hidden_dim, num_layers, dropout)
        self.attention = Attention(hidden_dim * 2)
        self.score_head = ScoreHead(hidden_dim * 2, dropout)

    def forward(self, ids, lengths):
        x = self.embedding(ids)
        lstm_out = self.encoder(x, lengths)
        context, weights = self.attention(lstm_out, lengths)
        score = self.score_head(context)
        return score, weights

bilstm_model = PairScorer(
    vocab_size=len(vocab), embed_dim=CONFIG["bilstm_embed_dim"], hidden_dim=CONFIG["bilstm_hidden_dim"],
    num_layers=CONFIG["bilstm_num_layers"], dropout=CONFIG["bilstm_dropout"], pad_idx=PAD_IDX,
    pretrained=embed_matrix, freeze=CONFIG["bilstm_freeze_embeddings"],
).to(device)

wandb.watch(bilstm_model, log="all", log_freq=50, idx=1)
print(bilstm_model)


PairScorer(
  (embedding): WordEmbedding(
    (embedding): Embedding(2973, 100, padding_idx=0)
  )
  (encoder): BiLSTMEncoder(
    (lstm): LSTM(100, 128, num_layers=2, batch_first=True, dropout=0.35, bidirectional=True)
  )
  (attention): Attention(
    (attn): Linear(in_features=256, out_features=256, bias=True)
    (v): Linear(in_features=256, out_features=1, bias=False)
  )
  (score_head): ScoreHead(
    (net): Sequential(
      (0): Linear(in_features=256, out_features=128, bias=True)
      (1): GELU(approximate='none')
      (2): Dropout(p=0.35, inplace=False)
      (3): Linear(in_features=128, out_features=1, bias=True)
    )
  )
)


In [13]:
# Cell 13 — Train Model B (BiLSTM), logging to W&B with a "bilstm/" prefix
def map3_score(logits, targets):
    top3 = torch.topk(logits, k=3, dim=1).indices
    score = 0.0
    for pred_row, true in zip(top3.tolist(), targets.tolist()):
        if true in pred_row:
            score += 1.0 / (pred_row.index(true) + 1)
    return score / len(targets)

bilstm_criterion = nn.CrossEntropyLoss(label_smoothing=CONFIG["bilstm_label_smoothing"])
bilstm_optimizer = torch.optim.AdamW(bilstm_model.parameters(), lr=CONFIG["bilstm_lr"], weight_decay=CONFIG["bilstm_weight_decay"])
bilstm_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(bilstm_optimizer, mode="max", factor=0.5, patience=3)
bilstm_scaler = torch.amp.GradScaler(device.type, enabled=(device.type == "cuda"))

def bilstm_run_epoch(loader, train=True):
    bilstm_model.train() if train else bilstm_model.eval()
    total_loss = total_map3 = 0.0
    n_batches = 0
    for ids, lengths, y in loader:
        b, n_opts, seq_len = ids.shape
        ids, lengths, y = ids.to(device), lengths.to(device), y.to(device)
        flat_ids, flat_lengths = ids.view(b * n_opts, seq_len), lengths.view(b * n_opts)

        with torch.set_grad_enabled(train):
            with torch.amp.autocast(device.type, enabled=(device.type == "cuda")):
                flat_scores, _ = bilstm_model(flat_ids, flat_lengths)
                logits = flat_scores.view(b, n_opts)
                loss = bilstm_criterion(logits, y)
            if train:
                bilstm_optimizer.zero_grad()
                bilstm_scaler.scale(loss).backward()
                bilstm_scaler.unscale_(bilstm_optimizer)
                torch.nn.utils.clip_grad_norm_(bilstm_model.parameters(), CONFIG["bilstm_grad_clip"])
                bilstm_scaler.step(bilstm_optimizer)
                bilstm_scaler.update()

        total_loss += loss.item()
        total_map3 += map3_score(logits.detach(), y)
        n_batches += 1
    return total_loss / n_batches, total_map3 / n_batches

best_bilstm_map3, patience_left = 0.0, CONFIG["bilstm_early_stop_patience"]

for epoch in range(1, CONFIG["bilstm_epochs"] + 1):
    train_loss, train_map3 = bilstm_run_epoch(bilstm_train_loader, train=True)
    val_loss, val_map3 = bilstm_run_epoch(bilstm_val_loader, train=False)
    bilstm_scheduler.step(val_map3)

    wandb.log({
        "bilstm/epoch": epoch, "bilstm/train_loss": train_loss, "bilstm/train_map3": train_map3,
        "bilstm/val_loss": val_loss, "bilstm/val_map3": val_map3,
        "bilstm/lr": bilstm_optimizer.param_groups[0]["lr"],
    })

    if val_map3 > best_bilstm_map3 + CONFIG["bilstm_early_stop_min_delta"]:
        best_bilstm_map3 = val_map3
        patience_left = CONFIG["bilstm_early_stop_patience"]
        torch.save(bilstm_model.state_dict(), "best_bilstm_model.pt")
    elif epoch >= CONFIG["bilstm_early_stop_min_epochs"]:
        patience_left -= 1

    print(f"[BiLSTM] Epoch {epoch:02d} | train_loss {train_loss:.4f} | val_loss {val_loss:.4f} val_map3 {val_map3:.4f} | patience {patience_left}")
    if epoch >= CONFIG["bilstm_early_stop_min_epochs"] and patience_left <= 0:
        print("Early stopping (BiLSTM).")
        break

bilstm_model.load_state_dict(torch.load("best_bilstm_model.pt"))
print("Best BiLSTM val MAP@3:", best_bilstm_map3)


[BiLSTM] Epoch 01 | train_loss 1.2247 | val_loss 0.5651 val_map3 0.9427 | patience 10
[BiLSTM] Epoch 02 | train_loss 0.4476 | val_loss 0.3506 val_map3 0.9672 | patience 10
[BiLSTM] Epoch 03 | train_loss 0.3315 | val_loss 0.3079 val_map3 0.9766 | patience 10
[BiLSTM] Epoch 04 | train_loss 0.3042 | val_loss 0.2922 val_map3 0.9797 | patience 10
[BiLSTM] Epoch 05 | train_loss 0.2892 | val_loss 0.2850 val_map3 0.9786 | patience 10
[BiLSTM] Epoch 06 | train_loss 0.2762 | val_loss 0.2764 val_map3 0.9786 | patience 10
[BiLSTM] Epoch 07 | train_loss 0.2675 | val_loss 0.2770 val_map3 0.9786 | patience 10
[BiLSTM] Epoch 08 | train_loss 0.2713 | val_loss 0.2790 val_map3 0.9786 | patience 10
[BiLSTM] Epoch 09 | train_loss 0.2618 | val_loss 0.2749 val_map3 0.9786 | patience 10
[BiLSTM] Epoch 10 | train_loss 0.2589 | val_loss 0.2728 val_map3 0.9786 | patience 10
[BiLSTM] Epoch 11 | train_loss 0.2583 | val_loss 0.2705 val_map3 0.9786 | patience 10
[BiLSTM] Epoch 12 | train_loss 0.2580 | val_loss 0.271

## 4. Turn both models' scores into comparable probabilities
The TF-IDF model outputs an independent `sigmoid` score per option (not a distribution — the 5
scores for one question don't have to sum to 1). The BiLSTM model already outputs a softmax
distribution over its 5 options. **To average them fairly, both need to be probability
distributions over the same 5 options first.** So: for every question, take that model's 5 raw
scores and run them through `softmax` — this is the normalization step, done once per model.


In [14]:
# Cell 14 — Get raw scores from both models on val + test, in the same long format
@torch.no_grad()
def tfidf_score_long(loader, meta_df):
    tfidf_model.eval()
    scores = []
    for batch in loader:
        Xw, Xc = batch[0], batch[1]
        Xw, Xc = Xw.to(device), Xc.to(device)
        logits = tfidf_model(Xw, Xc)
        scores.extend(torch.sigmoid(logits).float().cpu().tolist())
    out = meta_df[["qid", "option"]].copy()
    out["score"] = scores
    return out

@torch.no_grad()
def bilstm_score_long(loader, df, has_labels):
    bilstm_model.eval()
    rows = []
    for batch in loader:
        ids, lengths = batch[0], batch[1]
        b, n_opts, seq_len = ids.shape
        ids, lengths = ids.to(device), lengths.to(device)
        flat_scores, _ = bilstm_model(ids.view(b * n_opts, seq_len), lengths.view(b * n_opts))
        logits = flat_scores.view(b, n_opts).cpu().numpy()
        rows.append(logits)
    all_logits = np.concatenate(rows, axis=0)   # (n_questions, 5)
    out_rows = []
    for i, qid in enumerate(df["id"].tolist()):
        for j, letter in enumerate(LABELS):
            out_rows.append({"qid": qid, "option": letter, "score": all_logits[i, j]})
    return pd.DataFrame(out_rows)

tfidf_val_scores  = tfidf_score_long(tfidf_val_loader, meta_val_tfidf)
tfidf_test_scores = tfidf_score_long(tfidf_test_loader, test_long[["qid", "option"]])

bilstm_val_scores  = bilstm_score_long(bilstm_val_loader, bilstm_val_part, has_labels=True)
bilstm_test_scores = bilstm_score_long(bilstm_test_loader, test_df, has_labels=False)


In [15]:
# Cell 15 — Softmax-normalize each model's scores within every question (qid group)
def softmax_within_question(df, score_col="score"):
    df = df.copy()
    df[score_col] = df.groupby("qid")[score_col].transform(
        lambda s: np.exp(s - s.max()) / np.exp(s - s.max()).sum()
    )
    return df

tfidf_val_prob   = softmax_within_question(tfidf_val_scores).rename(columns={"score": "tfidf_prob"})
bilstm_val_prob  = softmax_within_question(bilstm_val_scores).rename(columns={"score": "bilstm_prob"})
tfidf_test_prob  = softmax_within_question(tfidf_test_scores).rename(columns={"score": "tfidf_prob"})
bilstm_test_prob = softmax_within_question(bilstm_test_scores).rename(columns={"score": "bilstm_prob"})

val_probs  = tfidf_val_prob.merge(bilstm_val_prob, on=["qid", "option"])
val_probs  = val_probs.merge(tfidf_val_long[["qid", "option", "label"]], on=["qid", "option"])
test_probs = tfidf_test_prob.merge(bilstm_test_prob, on=["qid", "option"])

print("Model A alone -- val MAP@3:", round(compute_map3(val_probs.rename(columns={"tfidf_prob": "score"})), 5))
print("Model B alone -- val MAP@3:", round(compute_map3(val_probs.rename(columns={"bilstm_prob": "score"})), 5))


Model A alone -- val MAP@3: 0.99111
Model B alone -- val MAP@3: 0.98056


## 5. Ensemble — Level 1: weighted average of probabilities
`ensemble_score = alpha * tfidf_prob + (1 - alpha) * bilstm_prob`

`alpha = 1.0` is "TF-IDF only", `alpha = 0.0` is "BiLSTM only", `alpha = 0.5` is a plain average.
We try every `alpha` on a small grid and keep whichever gives the best **validation** MAP@3 — this
is the simplest possible ensemble and is fully explainable in one sentence for a viva.


In [16]:
# Cell 16 — Grid-search alpha on the validation set, log the curve to W&B
alphas = np.arange(0.0, 1.0 + 1e-9, CONFIG["alpha_grid_step"])
alpha_results = []

for alpha in alphas:
    val_probs["ensemble_score"] = alpha * val_probs["tfidf_prob"] + (1 - alpha) * val_probs["bilstm_prob"]
    map3 = compute_map3(val_probs.rename(columns={"ensemble_score": "score"}))
    alpha_results.append((alpha, map3))
    wandb.log({"ensemble/alpha": alpha, "ensemble/val_map3_at_alpha": map3})

best_alpha, best_ensemble_map3 = max(alpha_results, key=lambda t: t[1])
print(f"Best alpha: {best_alpha:.2f} | Best ensemble val MAP@3: {best_ensemble_map3:.5f}")

wandb.log({
    "ensemble/best_alpha": best_alpha,
    "ensemble/best_val_map3": best_ensemble_map3,
    "ensemble/tfidf_only_val_map3": best_tfidf_map3,
    "ensemble/bilstm_only_val_map3": best_bilstm_map3,
})


Best alpha: 0.55 | Best ensemble val MAP@3: 1.00000


In [17]:
# Cell 17 — Apply the best alpha to the test set and write the final submission
test_probs["ensemble_score"] = best_alpha * test_probs["tfidf_prob"] + (1 - best_alpha) * test_probs["bilstm_prob"]

pred_rows = []
for qid, g in test_probs.groupby("qid", sort=False):
    top3 = g.sort_values("ensemble_score", ascending=False)["option"].tolist()[:3]
    pred_rows.append({"ID": qid, "Prediction": " ".join(top3)})

submission = pd.DataFrame(pred_rows)
submission.to_csv("submission_ensemble.csv", index=False)
submission.head()


,ID,Prediction
0,1,A C D
1,2,B D C
2,3,B E D
3,4,E C D
4,5,C A D


In [18]:
# Cell 18 — Summary + close out the W&B run
print("Model A (TF-IDF) val MAP@3:   ", round(best_tfidf_map3, 5))
print("Model B (BiLSTM) val MAP@3:   ", round(best_bilstm_map3, 5))
print(f"Ensemble val MAP@3 (alpha={best_alpha:.2f}):", round(best_ensemble_map3, 5))

wandb.finish()


Model A (TF-IDF) val MAP@3:    0.99111
Model B (BiLSTM) val MAP@3:    0.97969
Ensemble val MAP@3 (alpha=0.55): 1.0


bilstm/epoch,▁▁▂▂▂▃▃▃▄▄▅▅▅▆▆▆▇▇▇██
bilstm/lr,███████▄▄▄▄▂▂▂▂▁▁▁▁▁▁
bilstm/train_loss,█▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
bilstm/train_map3,▁▇███████████████████
bilstm/val_loss,█▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
bilstm/val_map3,▁▆▇██████████████████
ensemble/alpha,▁▁▂▂▂▃▃▃▄▄▅▅▅▆▆▆▇▇▇██
ensemble/best_alpha,▁
ensemble/best_val_map3,▁
ensemble/bilstm_only_val_map3,▁
+7,...


## Notes for next steps
- **If the ensemble's val MAP@3 barely beats the better single model:** the two models are probably
  agreeing on most questions already (less "different mistakes" than hoped) — check the val set for
  questions where `tfidf_prob` and `bilstm_prob` disagree strongly and eyeball a few.
- **Strategy 1 (better ranking) is complementary to this notebook, not a replacement:** it improves
  *how* each model ranks its own options, which would also lift the ensemble once applied to either
  branch.
- **Going further:** the alpha grid-search here is per-run, global. A slightly stronger version
  learns a per-question weight (e.g. logistic regression on `[tfidf_prob, bilstm_prob]` as
  features with `label` as target) instead of one fixed alpha — same idea, one extra tiny model.
